In [5]:
import itertools
import math
import random
import time


def generate_cities(n, seed=42, size=100):
    random.seed(seed)
    return [(random.uniform(0, size), random.uniform(0, size)) for _ in range(n)]


def distance(city1, city2):
    return math.hypot(city1[0] - city2[0], city1[1] - city2[1])


def build_distance_matrix(cities):
    n = len(cities)
    dist = [[0.0] * n for _ in range(n)]

    for i in range(n):
        for j in range(n):
            if i != j:
                dist[i][j] = distance(cities[i], cities[j])

    return dist


def route_length(route, dist):
    total = 0.0

    for i in range(len(route)):
        total += dist[route[i]][route[(i + 1) % len(route)]]

    return total


def tsp_brute_force(dist):
    n = len(dist)
    cities = list(range(1, n))
    best_route, best_cost = None, float('inf')

    for perm in itertools.permutations(cities):
        route = (0,) + perm
        cost = route_length(route, dist)

        if cost < best_cost:
            best_cost, best_route = cost, route

    return list(best_route), best_cost


def tsp_nearest_neighbour(dist, start=0):
    n = len(dist)
    unvisited = set(range(n))
    route = [start]
    unvisited.remove(start)
    current = start

    while unvisited:
        nxt = min(unvisited, key=lambda city: dist[current][city])
        route.append(nxt)
        unvisited.remove(nxt)
        current = nxt

    return route, route_length(route, dist)


def tsp_dynamic_programming(dist):
    n = len(dist)
    all_visited = (1 << n) - 1
    dp = [[float('inf')] * n for _ in range(1 << n)]
    parent = [[-1] * n for _ in range(1 << n)]

    dp[1][0] = 0

    for mask in range(1 << n):
        if not (mask & 1):
            continue

        for last in range(n):
            if not (mask & (1 << last)) or dp[mask][last] == float('inf'):
                continue

            for nxt in range(n):
                if mask & (1 << nxt):
                    continue

                new_mask = mask | (1 << nxt)
                new_cost = dp[mask][last] + dist[last][nxt]

                if new_cost < dp[new_mask][nxt]:
                    dp[new_mask][nxt] = new_cost
                    parent[new_mask][nxt] = last

    best_cost, best_last = float('inf'), -1

    for last in range(1, n):
        cost = dp[all_visited][last] + dist[last][0]

        if cost < best_cost:
            best_cost, best_last = cost, last

    route = []
    mask, last = all_visited, best_last

    while last != -1:
        route.append(last)
        new_last = parent[mask][last]
        mask ^= (1 << last)
        last = new_last

    route.reverse()

    return route, best_cost


def analyse(n_values=(4, 6, 8, 10, 11)):
    print(f"{'N':>3} | {'Brute Force':>18} | {'Held-Karp (DP)':>18} | {'Nearest Neighbour':>20}")
    print("-" * 70)

    for n in n_values:
        cities = generate_cities(n)
        dist = build_distance_matrix(cities)

        if n <= 10:
            t0 = time.perf_counter()
            _, bf_cost = tsp_brute_force(dist)
            bf_time = time.perf_counter() - t0
            bf_str = f"{bf_cost:7.2f} ({bf_time * 1000:6.1f} ms)"
        else:
            bf_str = "skipped "

        t0 = time.perf_counter()
        _, dp_cost = tsp_dynamic_programming(dist)
        dp_time = time.perf_counter() - t0
        dp_str = f"{dp_cost:7.2f} ({dp_time * 1000:6.1f} ms)"

        t0 = time.perf_counter()
        _, nn_cost = tsp_nearest_neighbour(dist)
        nn_time = time.perf_counter() - t0

        gap = (nn_cost - dp_cost) / dp_cost * 100
        nn_str = f"{nn_cost:7.2f} ({nn_time * 1000:5.2f} ms, +{gap:4.1f}%)"

        print(f"{n:>3} | {bf_str:>18} | {dp_str:>18} | {nn_str:>20}")

    print("\nObservations:")
    print(" - Brute Force and Held-Karp DP always agree (both are exact/optimal).")
    print(" - Brute Force time explodes factorially -- unusable past ~10-11 cities.")
    print(" - Held-Karp DP is exact but still exponential (2^N) -- fine up to ~20 cities.")
    print(" - Nearest Neighbour is extremely fast but its route can be a few")
    print("   percent longer than the true optimum ('+X.X%' gap shown above).")

if __name__ == "__main__":

    cities = generate_cities(6)
    dist = build_distance_matrix(cities)

    print("Cities:")

    for i, c in enumerate(cities):
        print(f"City {i}: ({c[0]:.1f}, {c[1]:.1f})")

    bf_route, bf_cost = tsp_brute_force(dist)
    dp_route, dp_cost = tsp_dynamic_programming(dist)
    nn_route, nn_cost = tsp_nearest_neighbour(dist)

    print(f"\nBrute Force: {bf_route} -> {bf_cost:.2f}")
    print(f"Held-Karp DP: {dp_route} -> {dp_cost:.2f}")
    print(f"Nearest Neighbour: {nn_route} -> {nn_cost:.2f}")

    print("\nAnalysis:")
    analyse()

Cities:
City 0: (63.9, 2.5)
City 1: (27.5, 22.3)
City 2: (73.6, 67.7)
City 3: (89.2, 8.7)
City 4: (42.2, 3.0)
City 5: (21.9, 50.5)

Brute Force: [0, 3, 2, 5, 1, 4] -> 216.38
Held-Karp DP: [0, 4, 1, 5, 2, 3] -> 216.38
Nearest Neighbour: [0, 4, 1, 5, 2, 3] -> 216.38

Analysis:
  N |        Brute Force |     Held-Karp (DP) |    Nearest Neighbour
----------------------------------------------------------------------
  4 |  193.20 (   0.0 ms) |  193.20 (   0.1 ms) |  193.20 ( 0.02 ms, + 0.0%)
  6 |  216.38 (   0.3 ms) |  216.38 (   0.3 ms) |  216.38 ( 0.02 ms, + 0.0%)
  8 |  253.28 (  10.6 ms) |  253.28 (   1.8 ms) |  253.28 ( 0.03 ms, + 0.0%)
 10 |  264.14 ( 779.0 ms) |  264.14 (   8.6 ms) |  294.66 ( 0.04 ms, +11.6%)
 11 |           skipped  |  272.14 (  18.3 ms) |  305.37 ( 0.03 ms, +12.2%)

Observations:
 - Brute Force and Held-Karp DP always agree (both are exact/optimal).
 - Brute Force time explodes factorially -- unusable past ~10-11 cities.
 - Held-Karp DP is exact but still expone